# ME/CFS dataset quickstart
This notebook inspects the harmonized tables and performs an exploratory PCA on the NIH PBMC processed count matrix. It is not a diagnostic model and does not pool cohorts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA = ROOT / 'datasets' / 'harmonized'
sorted(p.name for p in DATA.glob('*.csv*'))[:10]


## Study metadata
Load each GEO sample table separately. Similar column names should not be treated as cross-study harmonization.

In [ ]:
pbmc_meta = pd.read_csv(DATA / 'GSE251872_sample_metadata.csv')
muscle_meta = pd.read_csv(DATA / 'GSE245661_sample_metadata.csv')
print('PBMC:', pbmc_meta.shape)
print('Muscle:', muscle_meta.shape)
pbmc_meta.head()


## PBMC RNA-seq matrix
The matrix contains repository-supplied processed counts. Duplicate handling is logged in `GSE251872_duplicate_row_log.csv`.

In [ ]:
counts = pd.read_csv(DATA / 'GSE251872_counts_wide.csv.gz')
sample_cols = [c for c in counts.columns if c not in {'gene_id','external_gene_name'}]
X = counts[sample_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
print(counts.shape, len(sample_cols))
counts[['gene_id','external_gene_name']].head()


## Exploratory log-CPM PCA
Confirm participant IDs, case/control labels, repeated measures, covariates and batch structure before inferential analysis.

In [ ]:
library = X.sum(axis=0).replace(0, np.nan)
log_cpm = np.log1p(X.div(library, axis=1) * 1_000_000)
variances = log_cpm.var(axis=1)
top = log_cpm.loc[variances.nlargest(min(2000, len(variances))).index].T
scores = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(top))
pca_df = pd.DataFrame(scores, index=top.index, columns=['PC1','PC2'])
ax = pca_df.plot.scatter('PC1','PC2', figsize=(7,5))
for name, row in pca_df.iterrows():
    ax.annotate(name, (row.PC1, row.PC2), fontsize=6, alpha=.7)
plt.title('Exploratory PBMC log-CPM PCA')
plt.show()


## Next analytical steps
1. Parse case/control, sex, age, illness duration and challenge timing from GEO metadata.
2. Map sample columns to participants and split data at participant level.
3. Prefer study-specific differential-expression and pathway models over unconstrained classifiers.
4. Treat imaging cohorts separately unless a formal harmonization model is specified.
5. Report effect sizes and uncertainty; sample sizes are small.